In [0]:
# MAGIC %md
# MAGIC # Capa Bronze: Homogenización de Formato
# MAGIC Este notebook toma los datos extraídos de la API y los almacena en formato Delta.

from pyspark.sql import functions as F

# 1. Definir el nombre de la tabla
TABLA_BRONZE = "telemetria_patio_bronze"

# 2. Verificación de existencia de datos
# (Asumiendo que los datos vienen del paso anterior o de la tabla ya creada)
if spark.catalog.tableExists(TABLA_BRONZE):
    df_raw = spark.table(TABLA_BRONZE)
    print(f"✅ Cargando datos existentes de {TABLA_BRONZE}")
else:
    # Si es la primera ejecución y no hay datos, lanzamos una alerta
    dbutils.notebook.exit("❌ La tabla Bronze no existe. Ejecuta primero el Notebook de Ingesta.")

# 3. Estructuración Básica (Bronze)
# En esta capa solo añadimos metadatos de auditoría, como la fecha de carga.
df_bronze = df_raw.withColumn("fecha_ingesta_bronze", F.current_timestamp()) \
                  .withColumn("nombre_archivo_origen", F.lit("api_telemetria_json"))

# 4. Persistencia en Formato Delta
# Usamos 'append' para ir acumulando el historial de telemetría
try:
    df_bronze.write.format("delta") \
             .mode("append") \
             .option("mergeSchema", "true") \
             .saveAsTable(TABLA_BRONZE)
    
    print(f"📦 Datos homogenizados y guardados en la tabla Delta: {TABLA_BRONZE}")
    
    # Mostrar esquema para confirmar que es Raw
    df_bronze.printSchema()
    display(df_bronze.limit(10))

except Exception as e:
    print(f"❌ Error al guardar en Capa Bronze: {e}")